# Recurrent Neural Networks (LSTM/GRU)

RNNs process sequential data by maintaining hidden state across time steps.

1. **Vanilla RNN** - Vanishing gradient problem
2. **LSTM** - Long Short-Term Memory with gates
3. **GRU** - Gated Recurrent Unit (simplified LSTM)
4. **Sequence Classification** - Sentiment analysis on IMDB-style data

**Task**: Character-level name generation (many-to-one classification)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_20newsgroups
from collections import Counter

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## LSTM Architecture

LSTM solves the vanishing gradient problem with three gates:

- **Forget gate**: What to discard from cell state
- **Input gate**: What new information to store
- **Output gate**: What to output from cell state

$$f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f) \quad \text{(forget)}$$
$$i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + b_i) \quad \text{(input)}$$
$$o_t = \sigma(W_o \cdot [h_{t-1}, x_t] + b_o) \quad \text{(output)}$$

In [ ]:
# Simple text classification with LSTM
# Use 20 newsgroups as our text dataset
categories = ["sci.space", "rec.sport.baseball"]
data = fetch_20newsgroups(subset="all", categories=categories,
                          remove=("headers", "footers", "quotes"))
texts, labels = data.data, data.target

# Simple character-level tokenization
chars = sorted(set("".join(texts[:500])))
char_to_idx = {c: i + 1 for i, c in enumerate(chars)}  # 0 reserved for padding
vocab_size = len(char_to_idx) + 1

def encode_text(text, max_len=200):
    encoded = [char_to_idx.get(c, 0) for c in text[:max_len]]
    padded = encoded + [0] * (max_len - len(encoded))
    return padded

X = torch.tensor([encode_text(t) for t in texts], dtype=torch.long)
y = torch.tensor(labels, dtype=torch.long)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

train_loader = DataLoader(list(zip(X_train, y_train)), batch_size=32, shuffle=True)
test_loader = DataLoader(list(zip(X_test, y_test)), batch_size=64, shuffle=False)

print(f"Vocab size: {vocab_size}, Train: {len(X_train)}, Test: {len(X_test)}")

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128, num_classes=2, num_layers=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim, num_layers=num_layers,
            batch_first=True, dropout=dropout, bidirectional=True
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, num_classes),  # *2 for bidirectional
        )
    
    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, (hidden, _) = self.lstm(embedded)
        # Concatenate final forward and backward hidden states
        hidden_cat = torch.cat((hidden[-2], hidden[-1]), dim=1)
        return self.classifier(hidden_cat)

model = LSTMClassifier(vocab_size).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Training
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(10):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Gradient clipping
        optimizer.step()
        total_loss += loss.item() * len(y_batch)
        correct += (logits.argmax(1) == y_batch).sum().item()
        total += len(y_batch)
    
    # Evaluate
    model.eval()
    test_correct, test_total = 0, 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits = model(X_batch)
            test_correct += (logits.argmax(1) == y_batch).sum().item()
            test_total += len(y_batch)
    
    print(f"Epoch {epoch+1:2d} | Loss: {total_loss/total:.4f} | "
          f"Train Acc: {correct/total:.4f} | Test Acc: {test_correct/test_total:.4f}")

## Key Takeaways

1. **LSTM > vanilla RNN** for sequences >20 tokens (vanishing gradient)
2. **Bidirectional LSTM** sees both past and future context - better for classification
3. **Gradient clipping** is essential to prevent exploding gradients in RNNs
4. **GRU is simpler** (2 gates vs 3) and often performs comparably to LSTM
5. **Transformers have largely replaced RNNs** for NLP, but LSTMs remain useful for time series